# World Models 2018

这里，我们将复现经典论文：[World Models](https://arxiv.org/abs/1803.10122). 整个模型结构较为清晰，只需理解 Vision (V), Memory (M), 和 Controller (C) 这三个模块，整个模型就差不多已经理解了。

简单说一下整体架构。Vision 部分由 VAE 实现，负责将观测图像压缩到隐空间；Memory 部分由 MDN-RNN 实现，是整个世界模型的动力学模型（dynamic model），负责模拟隐空间中状态量的变化；Controller 是个简单的 MLP，负责根据当前 Memory 状态与当前 Vision 数据进行动作决策。我们预期在实现基本框架后在 `CarRacing` 上面跑一跑，不涉及 `VizDoom` 相关实验。

## 1. 基本模块代码实现

这部分，我们分别实现 V, M, C 三个组件的代码。

### 1.1 Vision (VAE)

先来导入一些基本的库：

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

下面是 Encoder。作者将 `64 * 64 * 3` 的输入经过四层卷积，转化为了 `2 * 2 * 256` 的数据，然后用一个线性层映射到 $\mu$ 和 $\sigma$. 每个卷积层中卷积核 `size` 均为 `4`，步长为 `2`. 

注意，`Conv2d` 要求输入符合 $(B,C,H,W)$，如果输入 `X` 不是这个样子的（比如 $(B, H, W, C)$），需要额外进行一步操作（比如 `X = X.permute(0, 3, 1, 2)`）.

In [3]:
class Encoder(nn.Module):
    def __init__(self, latent_dim=32):
        super().__init__()
        self.latent_dim = latent_dim
        self.conv_layer = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=4, stride=2, padding=0), # -> (32, 31, 31)
            nn.ReLU(),

            nn.Conv2d(32, 64, kernel_size=4, stride=2, padding=0), # -> (64, 14, 14)
            nn.ReLU(),

            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=0), # -> (128, 6, 6)
            nn.ReLU(),

            nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=0), # -> (256, 2, 2)
            nn.ReLU(),
        )

        self.fc_mu = nn.Linear(256 * 2 * 2, latent_dim)
        self.fc_logvar = nn.Linear(256 * 2 * 2, latent_dim)

    def forward(self, X):
        # X: (batch_size, 3, 64, 64)
        X = self.conv_layer(X)
        X_flatten = X.flatten(start_dim=1)

        mu = self.fc_mu(X_flatten)
        logvar = self.fc_logvar(X_flatten)
        return mu, logvar

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

紧接着是 Decoder，如下：

In [4]:
class Decoder(nn.Module):
    def __init__(self, latent_dim=32):
        super().__init__()
        
        self.fc = nn.Linear(latent_dim, 1024 * 1 * 1) # z -> 1024 * 1 * 1 
        
        self.deconv_layers = nn.Sequential(
            # (1024, 1, 1) -> (128, 5, 5)
            nn.ConvTranspose2d(1024, 128, kernel_size=5, stride=2, padding=0),
            nn.ReLU(),

            # (128, 5, 5) -> (64, 13, 13)
            nn.ConvTranspose2d(128, 64, kernel_size=5, stride=2, padding=0),
            nn.ReLU(),

            # (64, 13, 13) -> (32, 30, 30)
            nn.ConvTranspose2d(64, 32, kernel_size=6, stride=2, padding=0),
            nn.ReLU(),

            # (32, 30, 30) -> (3, 64, 64)
            nn.ConvTranspose2d(32, 3, kernel_size=6, stride=2, padding=0),
            nn.Sigmoid()
        )
    
    def forward(self, z):
        x = self.fc(z)
        x = x.view(-1, 1024, 1, 1)
        x = self.deconv_layers(x)
        return x

下面进行一个简单的测试，以确保 `Encoder` 和 `Decoder` 的参数配置及张量维度均无误。

In [5]:
enc = Encoder()
dec = Decoder()

test_X = torch.randn(8, 3, 64, 64)

test_mu, test_logvar = enc(test_X)
test_z = enc.reparameterize(test_mu, test_logvar)
test_X_recon = dec(test_z)

print(f"Shape of test_z: {test_z.shape}")
print(f"Shape of reconstructed X: {test_X_recon.shape}")

Shape of test_z: torch.Size([8, 32])
Shape of reconstructed X: torch.Size([8, 3, 64, 64])


看来张量维度没有问题，VAE 这部分也就基本实现了，我们下面先搞个简单的，即 `Controller`，最后再来搞 `Memory` 部分。

### 1.2 Controller

`Controller` 接收来自 `Memory` 的隐藏向量 `h` 和 `Vision` 的潜在表示 `z` 作为输入，（在 `CarRacing` 中），输出油门力度 $[0,1]$，刹车力度 $[0,1]$，以及方向盘 $[-1,1]$。

没有隐藏层，整体就是一个线性的映射。不过我们需要还需要分别施加适当的激活函数，将输出限制在对应范围内。

In [6]:
class Controller(nn.Module):
    def __init__(self, z_dim=32, hidden_dim=256, action_dim=3):
        super().__init__()
        self.fc = nn.Linear(z_dim + hidden_dim, action_dim)

    def forward(self, z, h):
        cat_in = torch.cat([z, h], dim=1)

        # actions[0] for steer wheel, [1] for pedal, [2] for brake
        raw_actions = self.fc(cat_in)

        steer = torch.tanh(raw_actions[:, 0:1])
        gas = torch.sigmoid(raw_actions[:, 1:2])
        brake = torch.sigmoid(raw_actions[:, 2:3])
        actions = torch.cat([steer, gas, brake], dim=1)

        return actions

再次进行一个小测试：

In [8]:
controller = Controller()
test_z = torch.randn(8, 32)
test_h = torch.randn(8, 256)
actions = controller(test_z, test_h)
print(f"Shape of actions: {actions.shape}")
print(f"steer: {actions[0][0]}, pedal: {actions[0][1]}, brake: {actions[0][2]}")

Shape of actions: torch.Size([8, 3])
steer: 0.39368048310279846, pedal: 0.5557169318199158, brake: 0.3104011118412018


### 1.3 Memory

`Memory` 的整体架构是 MDN-RNN，RNN 由 LSTM 具体实现。整个架构如下图所示。

![mdn-rnn-arch](assets/mdn-rnn-arch.svg)

这里简单说明一下 MDN-RNN，其与正常的 RNN 基本相同，只是在输出层上，RNN 输出一个具体的向量，而 MDN-RNN 输出几组 $(\pi, \mu, \sigma)$，其中，$\pi$ 为当前分布的权重，$\mu$ 和 $\sigma$ 则表示一个概率分布。论文中，取组数 $K = 5$。得到 $\{(\pi, \mu, \sigma)\}$ 后，通过下面代码中的方法取样，并重构 $z'$。

In [23]:
import math
import numpy as np

In [15]:
class MDNRNN(nn.Module):
    def __init__(self, z_dim=32, hidden_dim=256, action_dim=3, num_gaussians=5, tau=1.0):
        """
        MDN-RNN
        """
        super().__init__()
        self.z_dim = z_dim
        self.hidden_dim = hidden_dim
        self.num_gaussians = num_gaussians
        self.tau = 1.0
        
        self.lstm = nn.LSTM(
            input_size=z_dim + action_dim, 
            hidden_size=hidden_dim, 
            num_layers=1, 
            batch_first=True
        )
        
        # predict pi, mu and sigma
        self.fc_pi = nn.Linear(hidden_dim, num_gaussians)
        self.fc_mu = nn.Linear(hidden_dim, num_gaussians * z_dim)
        self.fc_sigma = nn.Linear(hidden_dim, num_gaussians * z_dim)

    def forward(self, z_seq, action_seq, hidden=None):
        B, T, _ = z_seq.shape
        
        lstm_input = torch.cat([z_seq, action_seq], dim=-1) # (B, T, 32 + 3)
        
        output, hidden = self.lstm(lstm_input, hidden)     # (B, T, 256)
        
        pi_logits = self.fc_pi(output)                     # (B, T, K)
        pi = F.softmax(pi_logits / self.tau, dim=-1)
        
        mu = self.fc_mu(output).reshape(B, T, self.num_gaussians, self.z_dim) # (B, T, K, 32)
        
        sigma = self.fc_sigma(output).reshape(B, T, self.num_gaussians, self.z_dim) # (B, T, K, 32)
        sigma = torch.exp(sigma) # ensure sigma > 0
        
        return pi, mu, sigma, hidden

    def sample_mdn(self, pi, mu, sigma):

        k = torch.distributions.Categorical(probs=pi).sample() # (B, T)
    
        k = k[..., None, None] # (B, T, 1, 1)
        k = k.expand(-1, -1, 1, mu.size(-1)) # (B, T, 1, D)
    
        selected_mu = torch.gather(mu, dim=2, index=k).squeeze(2) # (B, T, D)
        selected_sigma = torch.gather(sigma, dim=2, index=k).squeeze(2) # (B, T, D)
    
        eps = torch.randn_like(selected_mu)
        z_recon = selected_mu + selected_sigma * math.sqrt(self.tau) * eps
    
        return z_recon

In [16]:
B = 8
T = 8
z_seq = torch.randn(B, T, 32)
actions = torch.randn(B, T, 3)

mdn_rnn = MDNRNN()

pi, mu, sigma, hidden = mdn_rnn(z_seq, actions)

print(f"pi: {pi.shape}")
print(f"mu: {mu.shape}")
print(f"sigma: {sigma.shape}")
print(f"hidden state 0: {hidden[0].shape}")
print(f"hidden state 1: {hidden[1].shape}")

z_recon = mdn_rnn.sample_mdn(pi, mu, sigma)
print(f"z_recon shape: {z_recon.shape}")

pi: torch.Size([8, 8, 5])
mu: torch.Size([8, 8, 5, 32])
sigma: torch.Size([8, 8, 5, 32])
hidden state 0: torch.Size([1, 8, 256])
hidden state 1: torch.Size([1, 8, 256])
z_recon shape: torch.Size([8, 8, 32])


另外，我注意到，官方的 `mdn-rnn` 实现中直接使用一个线性头将 `hidden state` 映射到 `[pi, mu, sigma]` 一整个向量中，而不是像我这样搞成三个线性头。官方实现具有更优的效率，但我还是保留我最开始的做法。

接下来开始真实环境的训练了。

---

## 2. 环境准备与数据收集

首先，我们需要安装一些基本环境。另外，由于原来的 `CarRacing-v0` 版本过低，此处采用 `CarRacing-v3` 作为环境。

In [ ]:
!pip install "gymnasium[box2d]"
!pip install cma

安装完成后，开始准备环境并写一个数据收集脚本。

In [20]:
import gymnasium as gym

env = gym.make("CarRacing-v3", continuous=True)

按照 `gymnasium` 的标准流程，我们模拟汽车运动，使用随机策略，然后不断收集运动中的 `observations`, `actions` 数据，如下：

In [26]:
def collect_episode(env, max_steps=1000):
    obs, _ = env.reset()
    observations = [obs]
    actions = []
    rewards = []

    for t in range(max_steps):
        action = env.action_space.sample()

        next_obs, reward, terminated, truncated, _ = env.step(action)

        actions.append(action)
        rewards.append(reward)
        observations.append(next_obs)

        obs = next_obs

        if terminated or truncated:
            break

    return (
        np.asarray(observations, dtype=np.uint8),
        np.asarray(actions, dtype=np.float32),
        np.asarray(rewards, dtype=np.float32),
    )

In [27]:
observations, actions, rewards = collect_episode(env)

In [28]:
print(observations.shape, actions.shape, rewards.shape)

(1001, 96, 96, 3) (1000, 3) (1000,)


这里使用小规模的数据用于测试，官方一共跑了 `10000 episodes`，我们这里只收集了一个 episode 的数据。后续训练 VAE 的时候可能会考虑多跑点。

---

## 3. Vision 部分的训练

注意到，原始的输入是 $96\times 96 \times 3$ 的图像，我们需要转化后才可以输入到 VAE 中，具体流程如下：

$$
96\times96\times3
\rightarrow
3\times 64\times64
\rightarrow
VAE
\rightarrow
z\in\mathbb R^{32}.
$$

故而搞一个数据放缩函数：

In [34]:
def preprocess_obs(obs):
    # obs: (batch_size, 96, 96, 3)

    x = obs.float() / 255.0
    x = x.permute(0, 3, 1, 2)       # (batch_size, 3, 96, 96)

    x = F.interpolate(
        x,
        size=(64, 64),
        mode="bilinear",
        align_corners=False
    )

    return x

我们跑 10 个 epsisode，然后来跑模型。

In [32]:
observation_list = []
for _ in range(10):
    obs, _, _ = collect_episode(env)
    observation_list.append(torch.from_numpy(obs))
observations = torch.cat(observation_list, dim=0)

In [35]:
observations = preprocess_obs(observations)
print(observations.shape)

torch.Size([10010, 3, 64, 64])


In [36]:
enc_model = Encoder()
dec_model = Decoder()